In [1]:
from tools.scan_repo import scan_repo
from tools.read_file import read_file
from tools.write_file import write_file
from tools.notebook_sandbox import NotebookSandbox
from tools.record_attempt import record_attempt
from tools.execute_file import execute_file

from dotenv import load_dotenv
import re

import litellm

load_dotenv()
sandbox = NotebookSandbox()
litellm.success_callback = ["langsmith"]
litellm.failure_callback = ["langsmith"]
client = litellm.LiteLLM()

[SANDBOX] Initializing persistent Python kernel...
[SANDBOX] Isolated playground environment ready.


In [2]:
#for now don't focus on writing the code back into the file, instead make it output the report along with the buggy code.

In [3]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gemini/gemini-2.5-flash", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content

In [4]:
system_prompt = """
You are a Senior System Architect. Your objective is to debug and fix code by running a continuous Reasoning and Acting (ReAct) loop. 

You will operate in a strict cycle of: `Thought` -> `Action` -> `PAUSE` -> Wait for `Observation`. 
Once you have solved the bug (or failed after multiple attempts), you will output a final `Answer`.

### CRITICAL RULES (FAILURE TO FOLLOW THESE WILL BREAK THE SYSTEM):
1. **The PAUSE Keyword:** You MUST output the exact word `PAUSE` on a new line immediately after every `Action`. Stop generating text after `PAUSE`.
2. **No Empty Arguments:** If an Action requires no arguments, you MUST explicitly pass the string `None`. (e.g., `Action: scan_repo: None`).
3. **Python Code Formatting:** Any time you write Python code for a tool, you MUST wrap it exactly in these custom XML tags:
   <execute_python>
   # your full code here
   </execute_python>
4. **Never fake an Observation:** You must wait for the system to provide the `Observation:`. Never generate it yourself.
5. **No Shortcuts:** Write the complete code when fixing files or using the sandbox. Do not use placeholders like `[fix_code]`.

---

### AVAILABLE TOOLS

**File System Tools:**
* `scan_repo`: Scans the target repository and returns a list of all file names.
  * *Usage:* `Action: scan_repo: None`
* `read_file`: Reads and returns the entire content of a file (including jupyter notebooks).
  * *Usage:* `Action: read_file: file_name.ext`
* `write_file`: Writes full content into a file.
  * *Usage:* `Action: write_file: file_name.ext, <content>`
* `execute_file`: Executes a file and returns its stdout. Default timeout is 60s, but you can pass a custom integer if needed.
  * *Usage:* `Action: execute_file: file_name.ext, 60`

**Sandbox Tools (The sandbox is already initialized):**
* `notebook_sandbox`: Initializes or pings the jupyter notebook sandbox environment.
  * *Usage:* `Action: notebook_sandbox: None`
* `add_cell`: Appends a new Python code cell to the bottom of the sandbox. Returns the cell_id. 
  * *Usage:* `Action: add_cell: <execute_python> code </execute_python>`
* `execute_cell`: Executes a specific code cell by its ID. Variables/functions persist in memory.
  * *Usage:* `Action: execute_cell: cell_id`
* `edit_cell`: Overwrites the Python code in an existing cell to fix syntax/runtime errors.
  * *Usage:* `Action: edit_cell: cell_id, <execute_python> new_code </execute_python>`
* `get_sandbox_state`: Reads the entire virtual notebook. Returns all current cells, IDs, and last outputs.
  * *Usage:* `Action: get_sandbox_state: None`

**Tracking Tools:**
* `record_attempt`: Records your attempts to solve the bug.
  * *Usage:* `Action: record_attempt: hypothesis, code_changed, result, why_it_failed` (If successful, pass `None` for why_it_failed).

---

### EXAMPLE SESSION

user_question: Application is crashing with a "TypeError: 'NoneType' object is not subscriptable" error.

Thought: I need to find the files in this repository to locate the error.
Action: scan_repo: None
PAUSE

Observation: ['data_processor.py']

Thought: Given the repository structure, 'data_processor.py' is likely the file producing the error. I will read it.
Action: read_file: data_processor.py
PAUSE

Observation: [file content returned here...]

Thought: I see the bug. The sort method returns None. I will write the fix in the sandbox first to test it. 
Action: add_cell: 
<execute_python>
def get_highest_score(scores):
    scores.sort()
    return scores[-1]
    
math_scores = [85, 92, 78, 99, 88]
print(f"Highest: {get_highest_score(math_scores)}")
</execute_python>
PAUSE

Observation: Successfully added Code Cell 1.

Thought: Now I will execute the cell to verify my fix.
Action: execute_cell: 1
PAUSE

Observation: 99

Thought: Output confirms that the fix is working. I will reflect the changes back into the main file.
Action: write_file: data_processor.py, 
<execute_python>
def get_average_score(scores):
    if not scores:
        return 0
    total = sum(scores)
    return total / len(scores)

def get_highest_score(scores):
    scores.sort()
    return scores[-1]

if __name__ == "__main__":
    math_scores = [85, 92, 78, 99, 88]
    print(f"Average: {get_average_score(math_scores)}")
    print(f"Highest: {get_highest_score(math_scores)}")
</execute_python>
PAUSE

Observation: Successfully updated the file: data_processor.py

Thought: The bug is completely fixed and verified. I will output the final answer.
Answer: 
- Bug cause: [explain bug]
- Fix applied: [explain fix]
- How it works: [explain logic]
""".strip()

In [5]:
available_tools = {
    "read_file": read_file,
    "scan_repo": scan_repo,
    "write_file": write_file,
    "execute_file": execute_file,
    "notebook_sandbox": NotebookSandbox,
    "record_attempt": record_attempt,
    "add_cell": sandbox.add_cell,
    "edit_cell": sandbox.edit_cell,
    "execute_cell": sandbox.execute_cell,
    "get_sandbox_state": sandbox.get_sandbox_state
}
#I want the LLM to record the attempts it tried using record_attempt so that it doesn't repeat the same method again and also LLM knows the attempts it tried in the previous loop.

In [6]:
action_re = re.compile(r"Action:\s*(\w+):\s*(.*)PAUSE", re.DOTALL)


In [7]:
response = """LLM: Thought: I have created the sandbox. Now I will add the corrected `calculate_discount` function to a cell and test it with the provided values (price=100, percent=10).
Action: add_cell: <execute_python>
def calculate_discount(price, percent):
    discount = price * percent / 100 # Corrected division
    return discount # Return the discount amount
print(calculate_discount(100, 10))
</execute_python>
PAUSE"""
tool = action_re.search(response)
func, func_arg = tool.groups()
print(func, func_arg)
if func_arg.strip() != "None":
    print(f"calling: {func} on {func_arg}")
else:
    print(f"calling: {func}")
    
        
# if tool is not None:
#     func, func_arg = action_re.search(response).groups()
#             #4. call the tool
#     if func in available_tools:
#         if func_arg is not None: 
#             print(f"calling: {func} on input {func_arg}")
#             result = available_tools[func](func_arg)
#                     #5. send the result back to the LLM
#             next_prompt = f"Observation: {result}"
#             print(next_prompt)
#         else:
#             print(f"calling: {func}")
#             result = available_tools[func]()
#             next_prompt = f"Observation: {result}"
#             print(next_prompt)
#     else:
#         next_prompt = f"tool {func} is not available"
#         print(next_prompt)

add_cell <execute_python>
def calculate_discount(price, percent):
    discount = price * percent / 100 # Corrected division
    return discount # Return the discount amount
print(calculate_discount(100, 10))
</execute_python>

calling: add_cell on <execute_python>
def calculate_discount(price, percent):
    discount = price * percent / 100 # Corrected division
    return discount # Return the discount amount
print(calculate_discount(100, 10))
</execute_python>



In [8]:
import inspect
def query(user_prompt: str, max_steps = 10):
    #1. initialize
    bot = Agent(system_prompt)
    next_prompt = user_prompt
    for i in range(1, max_steps+1):
        print(f"-------- step {i} --------")
        #2. call LLM on user prompt or tool result
        response = bot(next_prompt)
        print(f"LLM: {response}")
        #3. check for tools
        tool = action_re.search(response)
        
        if tool is not None:
            func, func_arg = tool.groups()
            func, func_arg = func, func_arg.strip() #temporary fix
            #4. call the tool
            if func in available_tools:
                expected_args = len(inspect.signature(available_tools[func]).parameters)
                
                if expected_args == 0:
                    print(f"calling: {func}")
                    result = available_tools[func]()
                else:
                    arg_re = re.compile(r"([\w.]+),\s*(.*)", re.DOTALL)
                    if arg_re.match(func_arg) is not None:
                        arg1, arg2 = arg_re.match(func_arg).groups()
                        print(f"calling: {func} on input {arg1, arg2}")
                        result = available_tools[func](arg1, arg2)
                    else:
                        print(f"calling: {func} on input {func_arg}")
                        result = available_tools[func](func_arg)
                    
                #5. send the result back to the LLM
                next_prompt = f"Observation: {result}"
                print(next_prompt)
            else:
                next_prompt = f"tool {func} is not available"
                print(next_prompt)
        else:
            #end the loop if LLM wants no tool to access which means it reached it conclusion
            return "END OF LOOP"

In [9]:
user_prompt = "I am getting 0 as discount value for 100 rupees when 10 percent of 100 is 10"
query(user_prompt)

-------- step 1 --------
LLM: Action: scan_repo: None
PAUSE
```

calling: scan_repo
Observation: ['calculate_discount.py', 'test.ipynb', 'test_folder', 'test_folder/test_file.txt']
-------- step 2 --------
LLM: Action: read_file: calculate_discount.py
PAUSE
```

calling: read_file on input calculate_discount.py
Observation: def calculate_discount(price, percent):
    # BUG: dividing by 10 instead of 100
    discount = price * percent / 10
    return price - discount


def apply_tax(price, tax_rate):
    return price * (1 + tax_rate)


print(calculate_discount(100, 10))
-------- step 3 --------
LLM: Action: notebook_sandbox: None
PAUSE
```

calling: notebook_sandbox
[SANDBOX] Initializing persistent Python kernel...
[SANDBOX] Isolated playground environment ready.
Observation: <tools.notebook_sandbox.NotebookSandbox object at 0x0000017159F02FD0>
-------- step 4 --------
LLM: Thought: The problem states that the discount value is 0 for 100 rupees when 10 percent of 100 should be 10. Look

'END OF LOOP'